In [78]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.options.plotting.backend = "plotly"
pd.options.display.float_format = lambda x: f"{x:,.4f}"

from utils.logger import get_logger
from loaders.bdf_api import get_bdf_series
from loaders.sg_pee import get_sg_pee_data
from config.settings import (
    BDF_BASE_URL,
    BDF_HEADERS,
    KEY_INFLATION,
    KEY_LIVRET_A,
    ISIN_PEE,
    DRIVER_PATH,
)

from core.accounts.regulated_savings_account_v2 import RegulatedSavingAccount
from core.accounts.listed_account_v2 import ListedAccount

logger = get_logger("Main")

In [79]:
def ensure_cache_dir(path: str = "cache"):
    """Ensure cache directory exists."""
    if not os.path.exists(path):
        os.makedirs(path)
        logger.info(f"Created cache directory at '{path}'")


def main():
    logger.info("=== Starting financial data load ===")
    ensure_cache_dir()

    data_sources = {
        "Livret A": {
            "func": get_bdf_series,
            "params": {
                "series_key": KEY_LIVRET_A,
                "base_url": BDF_BASE_URL,
                "headers": BDF_HEADERS,
                "start_date": "2020-01-01",
                "cache_path": "cache/livret_a.pkl",
            },
        },
        "Inflation": {
            "func": get_bdf_series,
            "params": {
                "series_key": KEY_INFLATION,
                "base_url": BDF_BASE_URL,
                "headers": BDF_HEADERS,
                "start_date": "2020-01-01",
                "cache_path": "cache/inflation.pkl",
            },
        },
        "PEE": {
            "func": get_sg_pee_data,
            "params": {
                "isin": ISIN_PEE,
                "driver_path": DRIVER_PATH,
                "headless": True,
                "cache_path": "cache/pee.pkl",
            },
        },
    }

    results = {}

    for label, info in data_sources.items():
        try:
            logger.info(f"Loading {label} data...")
            df = info["func"](**info["params"])
            logger.info(f"{label} data loaded successfully ({len(df)} rows)")
            results[label] = df
        except Exception as e:
            logger.error(f"Failed to load {label} data: {e}")
            results[label] = None

    logger.info("=== Financial data load completed ===")

    return results




In [80]:
'''
def clear_cache(cache_name=None):
    """Supprime le cache. Si cache_name=None, supprime tous les caches."""
    cache_dir = "cache"
    if cache_name:
        path = os.path.join(cache_dir, cache_name)
        if os.path.exists(path):
            os.remove(path)
            logger.info(f"Cache '{cache_name}' supprimé")
        else:
            logger.info(f"Cache '{cache_name}' non trouvé")
    else:
        for f in os.listdir(cache_dir):
            if f.endswith(".pkl"):
                os.remove(os.path.join(cache_dir, f))
        logger.info("Tous les caches ont été supprimés")

clear_cache("pee.pkl")   # Supprime uniquement le cache PEE
clear_cache()            # Supprime tous les caches
'''


'\ndef clear_cache(cache_name=None):\n    """Supprime le cache. Si cache_name=None, supprime tous les caches."""\n    cache_dir = "cache"\n    if cache_name:\n        path = os.path.join(cache_dir, cache_name)\n        if os.path.exists(path):\n            os.remove(path)\n            logger.info(f"Cache \'{cache_name}\' supprimé")\n        else:\n            logger.info(f"Cache \'{cache_name}\' non trouvé")\n    else:\n        for f in os.listdir(cache_dir):\n            if f.endswith(".pkl"):\n                os.remove(os.path.join(cache_dir, f))\n        logger.info("Tous les caches ont été supprimés")\n\nclear_cache("pee.pkl")   # Supprime uniquement le cache PEE\nclear_cache()            # Supprime tous les caches\n'

In [81]:

results = main()
df_la = results['Livret A']
df_i = results['Inflation']
df_pee = results['PEE']


[2026-05-17 14:56:51] [INFO] Main: === Starting financial data load ===
[2026-05-17 14:56:51] [INFO] Main: Loading Livret A data...
[2026-05-17 14:56:51] [INFO] Cache: [CACHE EXPIRED] cache/livret_a.pkl (age: 693s)
[2026-05-17 14:56:51] [INFO] BDF_API: [API CALL] Requesting Banque de France series 'MIR1.M.FR.B.L23FRLA.D.R.A.2230U6.EUR.O'
[2026-05-17 14:56:52] [INFO] Cache: [CACHE SAVE] Saved to cache/livret_a.pkl
[2026-05-17 14:56:52] [INFO] BDF_API: [CACHE SAVE] Series 'MIR1.M.FR.B.L23FRLA.D.R.A.2230U6.EUR.O' cached to cache/livret_a.pkl
[2026-05-17 14:56:52] [INFO] Main: Livret A data loaded successfully (75 rows)
[2026-05-17 14:56:52] [INFO] Main: Loading Inflation data...
[2026-05-17 14:56:52] [INFO] Cache: [CACHE EXPIRED] cache/inflation.pkl (age: 693s)
[2026-05-17 14:56:52] [INFO] BDF_API: [API CALL] Requesting Banque de France series 'HICP.M.FR.N.000000.4D0.INX'
[2026-05-17 14:56:52] [INFO] Cache: [CACHE SAVE] Saved to cache/inflation.pkl
[2026-05-17 14:56:52] [INFO] BDF_API: [C

In [82]:
df_data = pd.read_excel("data/investment.xlsx", index_col=[1, 2, 0], usecols=range(11))
today = datetime.today()
df_data[['other_fees', 'broker_fees', 'tax']] = df_data[['other_fees', 'broker_fees', 'tax']].fillna(0)
df_data['empty_date'] = pd.NaT # Date ou l'ensemble de la ligne a été soldée
df_data.sample(2)

,,,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date
account,ticker,date,,,,,,,,,
PEA,AI.PA,2025-06-09,25.0000,182.5200,4.1100,0.0000,NaN,18.2500,^FCHI,NaN,NaT
LR,Livret A,2024-05-06,NaN,"1,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT


In [83]:
dfl = (
    df_data.loc['LR'][["value", "empty_date", "indice"]]
    .sort_index(axis=0)
    .copy()
)
dfl["value_adj"] = dfl["value"]
dfl[["rl_interest_adj", "th_interest_adj"]] = 0.0
dfl

"""
# dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g))
la = FixedRateInvestment(dfl.loc['Livret A'])
ld = FixedRateInvestment(dfl.loc['LDDS'])
lj = FixedRateInvestment(dfl.loc['Livret jeune'])
"""

"\n# dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g))\nla = FixedRateInvestment(dfl.loc['Livret A'])\nld = FixedRateInvestment(dfl.loc['LDDS'])\nlj = FixedRateInvestment(dfl.loc['Livret jeune'])\n"

In [84]:
"""
dfl = (
    df_data.loc['LR'][["value", "empty_date", "indice"]]
    .sort_index(axis=0)
    .copy()
)
dfl["value_adj"] = dfl["value"]
dfl[["rl_interest_adj", "th_interest_adj"]] = 0.0
dfl

investments = dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g.name, g.droplevel(0)))
investments_timeline = pd.concat(investments.apply(lambda inv: inv.positions_timeline).to_dict(), axis=1)
investments_timeline.tail(3)
"""

'\ndfl = (\n    df_data.loc[\'LR\'][["value", "empty_date", "indice"]]\n    .sort_index(axis=0)\n    .copy()\n)\ndfl["value_adj"] = dfl["value"]\ndfl[["rl_interest_adj", "th_interest_adj"]] = 0.0\ndfl\n\ninvestments = dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g.name, g.droplevel(0)))\ninvestments_timeline = pd.concat(investments.apply(lambda inv: inv.positions_timeline).to_dict(), axis=1)\ninvestments_timeline.tail(3)\n'

In [85]:
df_data.sample(3)

count      value  broker_fees  other_fees  \
account ticker   date                                                     
PEA     AI.PA    2024-02-05  3.0000   166.5600       0.9900      0.0000   
        PINR.PA  2024-07-30 66.0000    29.4740       2.9000     -2.9000   
LR      Livret A 2024-09-27     NaN 2,000.0000       0.0000      0.0000   

                             annual_fee_rate    tax indice  abondement  \
account ticker   date                                                    
PEA     AI.PA    2024-02-05              NaN 1.5000  ^FCHI         NaN   
        PINR.PA  2024-07-30           0.0085 0.0000  ^NSEI         NaN   
LR      Livret A 2024-09-27              NaN 0.0000  ^FCHI         NaN   

                            empty_date  
account ticker   date                   
PEA     AI.PA    2024-02-05        NaT  
        PINR.PA  2024-07-30        NaT  
LR      Livret A 2024-09-27        NaT

In [86]:
df_data.loc['LR']

count       value  broker_fees  other_fees  \
ticker       date                                                     
LDDS         2023-12-31    NaN 12,226.8400       0.0000      0.0000   
Livret jeune 2023-12-31    NaN  2,054.5300       0.0000      0.0000   
Livret A     2023-12-31    NaN  8,675.9000       0.0000      0.0000   
             2024-04-15    NaN  3,000.0000       0.0000      0.0000   
             2024-05-06    NaN  1,000.0000       0.0000      0.0000   
             2024-07-02    NaN   -500.0000       0.0000      0.0000   
             2024-09-05    NaN  1,000.0000       0.0000      0.0000   
             2024-09-13    NaN  4,000.0000       0.0000      0.0000   
             2024-09-27    NaN  2,000.0000       0.0000      0.0000   
             2024-10-08    NaN  3,774.1000       0.0000      0.0000   
Livret jeune 2025-01-06    NaN -2,116.1600       0.0000      0.0000   

                         annual_fee_rate    tax indice  abondement empty_date  
ticker       date                                                              
LDDS         2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
Livret jeune 2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
Livret A     2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-04-15              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-05-06              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-07-02              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-05              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-13              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-27              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-10-08              NaN 0.0000  ^FCHI         NaN        NaT  
Livret jeune 2025-01-06              NaN 0.0000  ^FCHI         NaN        NaT

In [87]:
df_data.loc['LR']

count       value  broker_fees  other_fees  \
ticker       date                                                     
LDDS         2023-12-31    NaN 12,226.8400       0.0000      0.0000   
Livret jeune 2023-12-31    NaN  2,054.5300       0.0000      0.0000   
Livret A     2023-12-31    NaN  8,675.9000       0.0000      0.0000   
             2024-04-15    NaN  3,000.0000       0.0000      0.0000   
             2024-05-06    NaN  1,000.0000       0.0000      0.0000   
             2024-07-02    NaN   -500.0000       0.0000      0.0000   
             2024-09-05    NaN  1,000.0000       0.0000      0.0000   
             2024-09-13    NaN  4,000.0000       0.0000      0.0000   
             2024-09-27    NaN  2,000.0000       0.0000      0.0000   
             2024-10-08    NaN  3,774.1000       0.0000      0.0000   
Livret jeune 2025-01-06    NaN -2,116.1600       0.0000      0.0000   

                         annual_fee_rate    tax indice  abondement empty_date  
ticker       date                                                              
LDDS         2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
Livret jeune 2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
Livret A     2023-12-31              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-04-15              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-05-06              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-07-02              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-05              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-13              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-09-27              NaN 0.0000  ^FCHI         NaN        NaT  
             2024-10-08              NaN 0.0000  ^FCHI         NaN        NaT  
Livret jeune 2025-01-06              NaN 0.0000  ^FCHI         NaN        NaT

In [88]:
df = df_data.loc['LR'].loc['Livret jeune'][["value", "empty_date", "indice"]].sort_index().copy()
df["value_adj"] = df["value"]
df[["rl_interest_adj", "th_interest_adj"]] = 0.0
df

,value,empty_date,indice,value_adj,rl_interest_adj,th_interest_adj
date,,,,,,
2023-12-31,"2,054.5300",NaT,^FCHI,"2,054.5300",0.0000,0.0000
2025-01-06,"-2,116.1600",NaT,^FCHI,"-2,116.1600",0.0000,0.0000


In [89]:
df_data.loc['LR'].loc['Livret A']

,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date
date,,,,,,,,,
2023-12-31,NaN,"8,675.9000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-04-15,NaN,"3,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-05-06,NaN,"1,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-07-02,NaN,-500.0000,0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-09-05,NaN,"1,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-09-13,NaN,"4,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-09-27,NaN,"2,000.0000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
2024-10-08,NaN,"3,774.1000",0.0000,0.0000,NaN,0.0000,^FCHI,NaN,NaT


In [90]:
lr = RegulatedSavingAccount(df_data.loc['LR'])

[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Initializing fixed-rate investment for LDDS
[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Using cached interest rates.
[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Initializing fixed-rate investment for Livret A
[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Using cached interest rates.
[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Initializing fixed-rate investment for Livret jeune
[2026-05-17 14:56:54] [INFO] FixedRateInvestment: Using cached interest rates.
[2026-05-17 14:56:55] [INFO] RegulatedSavingAccount: Successfully built 3 regulated savings investments


In [91]:
lr.investments

LDDS            <core.investments.fixed_rate.FixedRateInvestme...
Livret jeune    <core.investments.fixed_rate.FixedRateInvestme...
Livret A        <core.investments.fixed_rate.FixedRateInvestme...
dtype: object

In [92]:
lr.investments_timeline.tail(4)

LDDS                  \
date                                        2023-12-31                   
                                             value_adj rl_interest_adj   
interval_index                                                           
[2026-04-01 00:00:00, 2026-04-15 00:00:00] 12,226.8400        638.6100   
[2026-04-16 00:00:00, 2026-04-30 00:00:00] 12,226.8400        638.6100   
[2026-05-01 00:00:00, 2026-05-15 00:00:00] 12,226.8400        638.6100   
[2026-05-16 00:00:00, 2026-05-31 00:00:00] 12,226.8400        638.6100   

                                                           Livret jeune  \
date                                                         2023-12-31   
                                           th_interest_adj    value_adj   
interval_index                                                            
[2026-04-01 00:00:00, 2026-04-15 00:00:00]         58.4306          NaN   
[2026-04-16 00:00:00, 2026-04-30 00:00:00]         66.4715          NaN   
[2026-05-01 00:00:00, 2026-05-15 00:00:00]          0.0000          NaN   
[2026-05-16 00:00:00, 2026-05-31 00:00:00]          0.0000          NaN   

                                                                            \
date                                                                         
                                           rl_interest_adj th_interest_adj   
interval_index                                                               
[2026-04-01 00:00:00, 2026-04-15 00:00:00]             NaN             NaN   
[2026-04-16 00:00:00, 2026-04-30 00:00:00]             NaN             NaN   
[2026-05-01 00:00:00, 2026-05-15 00:00:00]             NaN             NaN   
[2026-05-16 00:00:00, 2026-05-31 00:00:00]             NaN             NaN   

                                             Livret A                  \
date                                       2023-12-31                   
                                            value_adj rl_interest_adj   
interval_index                                                          
[2026-04-01 00:00:00, 2026-04-15 00:00:00] 8,183.2900        428.0300   
[2026-04-16 00:00:00, 2026-04-30 00:00:00] 8,183.2900        428.0300   
[2026-05-01 00:00:00, 2026-05-15 00:00:00] 8,183.2900        428.0300   
[2026-05-16 00:00:00, 2026-05-31 00:00:00] 8,183.2900        428.0300   

                                                                       \
date                                                       2024-04-15   
                                           th_interest_adj  value_adj   
interval_index                                                          
[2026-04-01 00:00:00, 2026-04-15 00:00:00]         39.1097 3,000.0000   
[2026-04-16 00:00:00, 2026-04-30 00:00:00]         44.4918 3,000.0000   
[2026-05-01 00:00:00, 2026-05-15 00:00:00]          0.0000 3,000.0000   
[2026-05-16 00:00:00, 2026-05-31 00:00:00]          0.0000 3,000.0000   

                                                                            \
date                                                                         
                                           rl_interest_adj th_interest_adj   
interval_index                                                               
[2026-04-01 00:00:00, 2026-04-15 00:00:00]        129.8800         14.2149   
[2026-04-16 00:00:00, 2026-04-30 00:00:00]        129.8800         16.1710   
[2026-05-01 00:00:00, 2026-05-15 00:00:00]        129.8800          0.0000   
[2026-05-16 00:00:00, 2026-05-31 00:00:00]        129.8800          0.0000   

                                                                       \
date                                       2024-05-06                   
                                            value_adj rl_interest_adj   
interval_index                                                          
[2026-04-01 00:00:00, 2026-04-15 00:00:00] 1,000.0000         40.7400   
[2026-04-16 00:00:00, 2026-04-30 00:00:00] 1,000.0000         40.7400  

In [93]:
lr.investments_timeline.iloc[-1].loc['Livret jeune'].unstack(1).iloc[:,:2].sum(axis=1)

date
2023-12-31    0
dtype: object

In [94]:
lr.investments_timeline.iloc[-1].xs('rl_interest_adj', level=2).groupby(level=0).sum()

LDDS           638.6100
Livret A       915.3100
Livret jeune          0
Name: [2026-05-16 00:00:00, 2026-05-31 00:00:00], dtype: object

In [95]:
lr.investments_closed_positions

,,value,rl_interest_adj,th_interest_adj,short_date,value_short,indice
ticker,date,,,,,,
Livret jeune,2023-12-31,"2,054.5300",61.6300,0.0000,2025-01-06,"2,116.1600",^FCHI
Livret A,2023-12-31,492.6100,0.0000,7.3900,2024-07-02,500.0000,^FCHI


### Listed

In [96]:
df_data_fs = pd.read_excel("data/investment.xlsx", sheet_name='rompu', index_col=[1,2,0])
df_data_fs.sample()


,,,value
account,ticker,date,
PEA,AI.PA,2024-06-10,47.7600


In [97]:
df_data.loc['PEA']

,,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date
ticker,date,,,,,,,,,
GLE.PA,2023-10-13,22.0000,21.8600,0.9900,0.0000,NaN,1.4400,^FCHI,NaN,NaT
AYV.PA,2023-10-31,78.0000,6.3300,0.9900,0.0000,NaN,1.4800,^FCHI,NaN,NaT
AI.PA,2024-02-05,3.0000,166.5600,0.9900,0.0000,NaN,1.5000,^FCHI,NaN,NaT
TEP.PA,2024-05-16,9.0000,110.7500,1.9000,0.0000,NaN,2.9900,^FCHI,NaN,NaT
AI.PA,2024-06-07,10.0000,186.3800,2.9000,0.0000,NaN,5.5900,^FCHI,NaN,NaT
PINR.PA,2024-07-30,66.0000,29.4740,2.9000,-2.9000,0.0085,0.0000,^NSEI,NaN,NaT
ESE.PA,2024-12-09,150.0000,28.8389,3.8000,0.0000,0.0013,0.0000,^GSPC,NaN,NaT
AYV.PA,2025-02-06,-78.0000,7.2100,1.9000,0.0000,NaN,0.0000,^FCHI,NaN,NaT
DCAM.PA,2025-03-20,"1,000.0000",4.8440,4.3600,-4.3600,0.0020,0.0000,^GSPC,NaN,NaT


In [98]:
la = ListedAccount(df_data.loc['PEA'], df_data_fs.loc['PEA'], force_refresh=True)

[2026-05-17 14:56:56] [INFO] ListedAccount: Preprocessing completed: 13 rows prepared
[2026-05-17 14:56:56] [INFO] MarketDataManager: Initializing with tickers: ['AI.PA', 'AYV.PA', 'DCAM.PA', 'ESE.PA', 'GLE.PA', 'PINR.PA', 'SU.PA', 'TEP.PA', '^FCHI', '^GSPC', '^NSEI']
[2026-05-17 14:56:56] [INFO] MarketDataManager: Downloading tickers ['AYV.PA', 'PINR.PA', 'DCAM.PA', 'TEP.PA', '^GSPC', '^NSEI', 'ESE.PA', 'AI.PA', 'SU.PA', 'GLE.PA', '^FCHI'] from 2000-01-01 to today
[2026-05-17 14:56:58] [INFO] Cache: [CACHE SAVE] Saved to cache/market_data\all_tickers.pkl
[2026-05-17 14:56:58] [INFO] MarketDataManager: Cache updated: 6836 rows for ['AYV.PA', 'PINR.PA', 'DCAM.PA', 'TEP.PA', '^GSPC', '^NSEI', 'ESE.PA', 'AI.PA', 'SU.PA', 'GLE.PA', '^FCHI']
[2026-05-17 14:56:58] [INFO] ListedAccount: Successfully initialized MarketDataManager with 11 tickers
[2026-05-17 14:56:58] [INFO] ListedAccount: Inflation rates resolved for 74 periods
[2026-05-17 14:56:58] [INFO] ListedAccount: Starting parallel cons

In [99]:
la.investments.iloc[0]._inflation_rates



time_period_end
2020-02-29    84.6000
2020-03-31    84.6600
2020-04-30    84.6400
2020-05-31    84.7800
2020-06-30    84.8500
               ...   
2025-11-30    99.9200
2025-12-31   100.0000
2026-01-31    99.5900
2026-02-28   100.3000
2026-03-31   101.4000
Name: obs_value, Length: 74, dtype: float64

In [100]:
la.investments_closed_positions


,,value_adj_long,fees_adj_long,short_date,count_short,value_short,fees_short,fs_adj,tax_adj,abond_adj,indice
ticker,date,,,,,,,,,,
AYV.PA,2023-10-31,6.3300,0.9900,2025-02-06,78.0000,7.2100,1.9000,0.0000,1.4800,NaN,^FCHI


In [101]:
"""
from core.market_data.manager_v3 import MarketDataManager
md = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())))
                    
md = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())) + ['MSFT'])
md.get_market_data('AAPL')
"""

"\nfrom core.market_data.manager_v3 import MarketDataManager\nmd = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())))\n                    \nmd = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())) + ['MSFT'])\nmd.get_market_data('AAPL')\n"

In [102]:
aux = la.investments_timeline.loc[:, la.investments_timeline.columns.get_level_values(2) == 'count_nm']
aux = aux['AI.PA']
aux.columns = [
    '_'.join(map(str, col)).strip('_')
    for col in aux.columns
]
aux.plot()

c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [103]:
la.investments_timeline.head(3)

AYV.PA                                                DCAM.PA  \
date       2023-10-31                                             2025-03-20   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2023-10-13        NaN       NaN       NaN      NaN     NaN    NaN        NaN   
2023-10-16        NaN       NaN       NaN      NaN     NaN    NaN        NaN   
2023-10-17        NaN       NaN       NaN      NaN     NaN    NaN        NaN   

                                                           ESE.PA            \
date                                                   2024-12-09             
           count_adj value_adj fees_adj tax_adj fs_adj   count_nm count_adj   
Date                                                                          
2023-10-13       NaN       NaN      NaN     NaN    NaN        NaN       NaN   
2023-10-16       NaN       NaN      NaN     NaN    NaN        NaN       NaN   
2023-10-17       NaN       NaN      NaN     NaN    NaN        NaN       NaN   

                                                  AI.PA                      \
date                                         2024-02-05                       
           value_adj fees_adj tax_adj fs_adj   count_nm count_adj value_adj   
Date                                                                          
2023-10-13       NaN      NaN     NaN    NaN        NaN       NaN       NaN   
2023-10-16       NaN      NaN     NaN    NaN        NaN       NaN       NaN   
2023-10-17       NaN      NaN     NaN    NaN        NaN       NaN       NaN   

                                                                            \
date                               2024-06-07                                
           fees_adj tax_adj fs_adj   count_nm count_adj value_adj fees_adj   
Date                                                                         
2023-10-13      NaN     NaN    NaN        NaN       NaN       NaN      NaN   
2023-10-16      NaN     NaN    NaN        NaN       NaN       NaN      NaN   
2023-10-17      NaN     NaN    NaN        NaN       NaN       NaN      NaN   

                                                                           \
date                      2025-06-09                                        
           tax_adj fs_adj   count_nm count_adj value_adj fees_adj tax_adj   
Date                                                                        
2023-10-13     NaN    NaN        NaN       NaN       NaN      NaN     NaN   
2023-10-16     NaN    NaN        NaN       NaN       NaN      NaN     NaN   
2023-10-17     NaN    NaN        NaN       NaN       NaN      NaN     NaN   

                                                                          \
date              2025-10-31                                               
           fs_adj   count_nm count_adj value_adj fees_adj tax_adj fs_adj   
Date                                                                       
2023-10-13    NaN        NaN       NaN       NaN      NaN     NaN    NaN   
2023-10-16    NaN        NaN       NaN       NaN      NaN     NaN    NaN   
2023-10-17    NaN        NaN       NaN       NaN      NaN     NaN    NaN   

                                                                     PINR.PA  \
date       2026-05-12                                             2024-07-30   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2023-10-13        NaN       NaN       NaN      NaN     NaN    NaN        NaN   
2023-10-16        NaN       NaN       NaN      NaN     NaN    NaN        NaN   
2023-10-17        NaN       NaN       NaN      NaN     NaN    NaN        NaN   

                                                           GLE.PA            \
date                                                   2023-10-13             
       

In [104]:
la.investments_timeline.tail(3)

AYV.PA                                                DCAM.PA  \
date       2023-10-31                                             2025-03-20   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2026-05-13        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   
2026-05-14        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   
2026-05-15        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   

                                                            ESE.PA            \
date                                                    2024-12-09             
            count_adj value_adj fees_adj tax_adj fs_adj   count_nm count_adj   
Date                                                                           
2026-05-13 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   
2026-05-14 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   
2026-05-15 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   

                                                  AI.PA                      \
date                                         2024-02-05                       
           value_adj fees_adj tax_adj fs_adj   count_nm count_adj value_adj   
Date                                                                          
2026-05-13   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   
2026-05-14   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   
2026-05-15   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   

                                                                             \
date                                2024-06-07                                
           fees_adj tax_adj  fs_adj   count_nm count_adj value_adj fees_adj   
Date                                                                          
2026-05-13   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   
2026-05-14   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   
2026-05-15   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   

                                                                           \
date                      2025-06-09                                        
           tax_adj fs_adj   count_nm count_adj value_adj fees_adj tax_adj   
Date                                                                        
2026-05-13  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   
2026-05-14  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   
2026-05-15  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   

                                                                          \
date              2025-10-31                                               
           fs_adj   count_nm count_adj value_adj fees_adj tax_adj fs_adj   
Date                                                                       
2026-05-13 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   
2026-05-14 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   
2026-05-15 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   

                                                                     PINR.PA  \
date       2026-05-12                                             2024-07-30   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2026-05-13    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   
2026-05-14    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   
2026-05-15    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   

                                                           GLE.PA            \
date                                                   2023-10-13       

### Démonstration

In [105]:
la.investments

AYV.PA     <core.investments.listed_v2.ListedInvestment o...
DCAM.PA    <core.investments.listed_v2.ListedInvestment o...
ESE.PA     <core.investments.listed_v2.ListedInvestment o...
AI.PA      <core.investments.listed_v2.ListedInvestment o...
PINR.PA    <core.investments.listed_v2.ListedInvestment o...
GLE.PA     <core.investments.listed_v2.ListedInvestment o...
TEP.PA     <core.investments.listed_v2.ListedInvestment o...
SU.PA      <core.investments.listed_v2.ListedInvestment o...
dtype: object

In [106]:
la.transactions

count    value  broker_fees  other_fees  \
ticker  date                                                      
AI.PA   2024-02-05     3.0000 166.5600       0.9900      0.0000   
        2024-06-07    10.0000 186.3800       2.9000      0.0000   
        2025-06-09    25.0000 182.5200       4.1100      0.0000   
        2025-10-31    27.0000 167.6000       4.0700      0.0000   
        2026-05-12    34.0000 175.4000       5.3700      0.0000   
AYV.PA  2023-10-31    78.0000   6.3300       0.9900      0.0000   
        2025-02-06   -78.0000   7.2100       1.9000      0.0000   
DCAM.PA 2025-03-20 1,000.0000   4.8440       4.3600     -4.3600   
ESE.PA  2024-12-09   150.0000  28.8389       3.8000      0.0000   
GLE.PA  2023-10-13    22.0000  21.8600       0.9900      0.0000   
PINR.PA 2024-07-30    66.0000  29.4740       2.9000     -2.9000   
SU.PA   2025-06-09    20.0000 224.6000       4.0400      0.0000   
TEP.PA  2024-05-16     9.0000 110.7500       1.9000      0.0000   

                    annual_fee_rate     tax indice  abondement empty_date  \
ticker  date                                                                
AI.PA   2024-02-05              NaN  1.5000  ^FCHI         NaN        NaT   
        2024-06-07              NaN  5.5900  ^FCHI         NaN        NaT   
        2025-06-09              NaN 18.2500  ^FCHI         NaN        NaT   
        2025-10-31              NaN 18.1000  ^FCHI         NaN        NaT   
        2026-05-12              NaN 23.8500  ^FCHI         NaN        NaT   
AYV.PA  2023-10-31              NaN  1.4800  ^FCHI         NaN        NaT   
        2025-02-06              NaN  0.0000  ^FCHI         NaN        NaT   
DCAM.PA 2025-03-20           0.0020  0.0000  ^GSPC         NaN        NaT   
ESE.PA  2024-12-09           0.0013  0.0000  ^GSPC         NaN        NaT   
GLE.PA  2023-10-13              NaN  1.4400  ^FCHI         NaN        NaT   
PINR.PA 2024-07-30           0.0085  0.0000  ^NSEI         NaN        NaT   
SU.PA   2025-06-09              NaN 17.9700  ^FCHI         NaN        NaT   
TEP.PA  2024-05-16              NaN  2.9900  ^FCHI         NaN        NaT   

                     count_nm  count_adj  value_adj  tax_adj  abond_adj  \
ticker  date                                                              
AI.PA   2024-02-05     3.0000     3.0000   166.5600   1.5000        NaN   
        2024-06-07    10.0000    10.0000   186.3800   5.5900        NaN   
        2025-06-09    25.0000    25.0000   182.5200  18.2500        NaN   
        2025-10-31    27.0000    27.0000   167.6000  18.1000        NaN   
        2026-05-12    34.0000    34.0000   175.4000  23.8500        NaN   
AYV.PA  2023-10-31    78.0000    78.0000     6.3300   1.4800        NaN   
        2025-02-06   -78.0000   -78.0000     7.2100   0.0000        NaN   
DCAM.PA 2025-03-20 1,000.0000 1,000.0000     4.8440   0.0000        NaN   
ESE.PA  2024-12-09   150.0000   150.0000    28.8389   0.0000        NaN   
GLE.PA  2023-10-13    22.0000    22.0000    21.8600   1.4400        NaN   
PINR.PA 2024-07-30    66.0000    66.0000    29.4740   0.0000        NaN   
SU.PA   2025-06-09    20.0000    20.0000   224.6000  17.9700        NaN   
TEP.PA  2024-05-16     9.0000     9.0000   110.7500   2.9900        NaN   

                    fees_adj  fs_adj  
ticker  date                          
AI.PA   2024-02-05    0.9900  0.0000  
        2024-06-07    2.9000  0.0000  
        2025-06-09    4.1100  0.0000  
        2025-10-31    4.0700  0.0000  
        2026-05-12    5.3700  0.0000  
AYV.PA  2023-10-31    0.9900  0.0000  
        2025-02-06    1.9000  0.0000  
DCAM.PA 2025-03-20    0.0000  0.0000  
ESE.PA  2024-12-09    3.8000  0.0000  
GLE.PA  2023-10-13    0.9900  0.0000  
PINR.PA 2024-07-30    0.0000  0.0000  
SU.PA   2025-06-09    4.0400  0.0000  
TEP.PA  2024-05-16    1.9000  0.0000

In [107]:
la.investments_closed_positions

,,value_adj_long,fees_adj_long,short_date,count_short,value_short,fees_short,fs_adj,tax_adj,abond_adj,indice
ticker,date,,,,,,,,,,
AYV.PA,2023-10-31,6.3300,0.9900,2025-02-06,78.0000,7.2100,1.9000,0.0000,1.4800,NaN,^FCHI


In [108]:
la.investments_timeline.tail(3)

AYV.PA                                                DCAM.PA  \
date       2023-10-31                                             2025-03-20   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2026-05-13        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   
2026-05-14        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   
2026-05-15        NaN       NaN       NaN      NaN     NaN    NaN 1,000.0000   

                                                            ESE.PA            \
date                                                    2024-12-09             
            count_adj value_adj fees_adj tax_adj fs_adj   count_nm count_adj   
Date                                                                           
2026-05-13 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   
2026-05-14 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   
2026-05-15 1,000.0000    4.8440   0.0000  0.0000 0.0000   150.0000  150.0000   

                                                  AI.PA                      \
date                                         2024-02-05                       
           value_adj fees_adj tax_adj fs_adj   count_nm count_adj value_adj   
Date                                                                          
2026-05-13   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   
2026-05-14   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   
2026-05-15   28.8389   3.8000  0.0000 0.0000     3.0000    3.0000  166.5600   

                                                                             \
date                                2024-06-07                                
           fees_adj tax_adj  fs_adj   count_nm count_adj value_adj fees_adj   
Date                                                                          
2026-05-13   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   
2026-05-14   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   
2026-05-15   0.9900  1.5000 47.7600    11.0000   11.0000  169.4364   2.9000   

                                                                           \
date                      2025-06-09                                        
           tax_adj fs_adj   count_nm count_adj value_adj fees_adj tax_adj   
Date                                                                        
2026-05-13  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   
2026-05-14  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   
2026-05-15  5.5900 0.0000    25.0000   25.0000  182.5200   4.1100 18.2500   

                                                                          \
date              2025-10-31                                               
           fs_adj   count_nm count_adj value_adj fees_adj tax_adj fs_adj   
Date                                                                       
2026-05-13 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   
2026-05-14 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   
2026-05-15 0.0000    27.0000   27.0000  167.6000   4.0700 18.1000 0.0000   

                                                                     PINR.PA  \
date       2026-05-12                                             2024-07-30   
             count_nm count_adj value_adj fees_adj tax_adj fs_adj   count_nm   
Date                                                                           
2026-05-13    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   
2026-05-14    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   
2026-05-15    34.0000   34.0000  175.4000   5.3700 23.8500 0.0000    66.0000   

                                                           GLE.PA            \
date                                                   2023-10-13       

In [109]:
la.tickers

Index(['AI.PA', 'AYV.PA', 'DCAM.PA', 'ESE.PA', 'GLE.PA', 'PINR.PA', 'SU.PA',
       'TEP.PA'],
      dtype='object', name='ticker')

In [110]:
la.stock_data_fs

,,value
ticker,date,
AI.PA,2024-06-10,47.7600


In [111]:
target = 'AI.PA'

In [112]:
la.investments.loc[target].market_data

Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2000-01-03,26.1866,27.2777,26.1242,26.3424,14.0918,"1,203,599.0000",0.0000,0.0000
2000-01-04,26.3424,26.2957,24.5031,25.1734,13.4664,"1,838,702.0000",0.0000,0.0000
2000-01-05,24.6278,25.0955,24.2537,24.9396,13.3413,"1,136,359.0000",0.0000,0.0000
2000-01-06,24.6122,27.2153,24.5655,26.7321,14.3003,"1,846,279.0000",0.0000,0.0000
2000-01-07,26.7321,27.9012,26.2645,26.4048,14.1251,"2,925,848.0000",0.0000,0.0000
...,...,...,...,...,...,...,...,...
2026-05-11,175.0000,176.2200,174.3000,176.0000,176.0000,"601,438.0000",0.0000,0.0000
2026-05-12,175.6200,176.9400,175.2200,175.8200,175.8200,"612,937.0000",0.0000,0.0000
2026-05-13,177.3200,177.6200,175.7000,177.0200,177.0200,"564,830.0000",0.0000,0.0000


In [113]:
la.investments.loc[target].name

'AI.PA'

In [114]:
la.investments.loc[target].transactions

,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date,count_nm,count_adj,value_adj,tax_adj,abond_adj,fees_adj,fs_adj
date,,,,,,,,,,,,,,,,
2024-02-05,3.0000,166.5600,0.9900,0.0000,NaN,1.5000,^FCHI,NaN,NaT,3.0000,3.0000,166.5600,1.5000,NaN,0.9900,47.7600
2024-06-07,10.0000,186.3800,2.9000,0.0000,NaN,5.5900,^FCHI,NaN,NaT,10.0000,11.0000,169.4364,5.5900,NaN,2.9000,0.0000
2025-06-09,25.0000,182.5200,4.1100,0.0000,NaN,18.2500,^FCHI,NaN,NaT,25.0000,25.0000,182.5200,18.2500,NaN,4.1100,0.0000
2025-10-31,27.0000,167.6000,4.0700,0.0000,NaN,18.1000,^FCHI,NaN,NaT,27.0000,27.0000,167.6000,18.1000,NaN,4.0700,0.0000
2026-05-12,34.0000,175.4000,5.3700,0.0000,NaN,23.8500,^FCHI,NaN,NaT,34.0000,34.0000,175.4000,23.8500,NaN,5.3700,0.0000


In [115]:
la.investments.loc[target].fractional_share_data

,value
date,
2024-06-10,47.7600


In [116]:
la.investments.loc[target].positions_timeline.tail(3)

date       2024-02-05                                              2024-06-07  \
             count_nm count_adj value_adj fees_adj tax_adj  fs_adj   count_nm   
Date                                                                            
2026-05-13     3.0000    3.0000  166.5600   0.9900  1.5000 47.7600    11.0000   
2026-05-14     3.0000    3.0000  166.5600   0.9900  1.5000 47.7600    11.0000   
2026-05-15     3.0000    3.0000  166.5600   0.9900  1.5000 47.7600    11.0000   

date                                                   2025-06-09            \
           count_adj value_adj fees_adj tax_adj fs_adj   count_nm count_adj   
Date                                                                          
2026-05-13   11.0000  169.4364   2.9000  5.5900 0.0000    25.0000   25.0000   
2026-05-14   11.0000  169.4364   2.9000  5.5900 0.0000    25.0000   25.0000   
2026-05-15   11.0000  169.4364   2.9000  5.5900 0.0000    25.0000   25.0000   

date                                         2025-10-31                      \
           value_adj fees_adj tax_adj fs_adj   count_nm count_adj value_adj   
Date                                                                          
2026-05-13  182.5200   4.1100 18.2500 0.0000    27.0000   27.0000  167.6000   
2026-05-14  182.5200   4.1100 18.2500 0.0000    27.0000   27.0000  167.6000   
2026-05-15  182.5200   4.1100 18.2500 0.0000    27.0000   27.0000  167.6000   

date                               2026-05-12                               \
           fees_adj tax_adj fs_adj   count_nm count_adj value_adj fees_adj   
Date                                                                         
2026-05-13   4.0700 18.1000 0.0000    34.0000   34.0000  175.4000   5.3700   
2026-05-14   4.0700 18.1000 0.0000    34.0000   34.0000  175.4000   5.3700   
2026-05-15   4.0700 18.1000 0.0000    34.0000   34.0000  175.4000   5.3700   

date                       
           tax_adj fs_adj  
Date                       
2026-05-13 23.8500 0.0000  
2026-05-14 23.8500 0.0000  
2026-05-15 23.8500 0.0000

In [117]:
la.investments.loc[target].metrics.keys()

dict_keys(['performance', 'risk_adjusted_performance', 'risk', 'positions'])

In [118]:
la.investments.loc[target].metrics['performance'].keys()

dict_keys(['total_returns', 'annualized_return', 'market_value', 'equity_invested', 'benefit', 'dividends_received', 'dividend_yield', 'annualized_dividend_yield', 'fractional_shares', 'estimated_etf_fees'])

In [119]:
pd.concat(la.investments.loc[target].metrics['positions']).unstack(0).tail(3)

date           2024-02-05                                                     \
           nominal_shares adjusted_shares long_value   fees  taxes subsidies   
Date                                                                           
2026-05-13         3.0000          3.0000   166.5600 0.9900 1.5000    0.0000   
2026-05-14         3.0000          3.0000   166.5600 0.9900 1.5000    0.0000   
2026-05-15         3.0000          3.0000   166.5600 0.9900 1.5000    0.0000   

date           2024-06-07                                                     \
           nominal_shares adjusted_shares long_value   fees  taxes subsidies   
Date                                                                           
2026-05-13        11.0000         11.0000   169.4364 2.9000 5.5900    0.0000   
2026-05-14        11.0000         11.0000   169.4364 2.9000 5.5900    0.0000   
2026-05-15        11.0000         11.0000   169.4364 2.9000 5.5900    0.0000   

date           2025-06-09                                                      \
           nominal_shares adjusted_shares long_value   fees   taxes subsidies   
Date                                                                            
2026-05-13        25.0000         25.0000   182.5200 4.1100 18.2500    0.0000   
2026-05-14        25.0000         25.0000   182.5200 4.1100 18.2500    0.0000   
2026-05-15        25.0000         25.0000   182.5200 4.1100 18.2500    0.0000   

date           2025-10-31                                                      \
           nominal_shares adjusted_shares long_value   fees   taxes subsidies   
Date                                                                            
2026-05-13        27.0000         27.0000   167.6000 4.0700 18.1000    0.0000   
2026-05-14        27.0000         27.0000   167.6000 4.0700 18.1000    0.0000   
2026-05-15        27.0000         27.0000   167.6000 4.0700 18.1000    0.0000   

date           2026-05-12                                                      
           nominal_shares adjusted_shares long_value   fees   taxes subsidies  
Date                                                                           
2026-05-13        34.0000         34.0000   175.4000 5.3700 23.8500    0.0000  
2026-05-14        34.0000         34.0000   175.4000 5.3700 23.8500    0.0000  
2026-05-15        34.0000         34.0000   175.4000 5.3700 23.8500    0.0000

In [120]:
pd.concat(la.investments.loc[target].metrics['performance']).unstack(0).tail(3)

date          2024-02-05                                                 \
           total_returns annualized_return market_value equity_invested   
Date                                                                      
2026-05-13        0.1877            0.0788     531.0600        502.1700   
2026-05-14        0.1993            0.0834     536.8800        502.1700   
2026-05-15        0.1830            0.0768     528.7200        502.1700   

date                                                   \
            benefit dividends_received dividend_yield   
Date                                                    
2026-05-13  94.2500            19.5000         0.0388   
2026-05-14 100.0700            19.5000         0.0388   
2026-05-15  91.9100            19.5000         0.0388   

date                                                                       \
           annualized_dividend_yield fractional_shares estimated_etf_fees   
Date                                                                        
2026-05-13                    0.0169           47.7600             0.0000   
2026-05-14                    0.0169           47.7600             0.0000   
2026-05-15                    0.0169           47.7600             0.0000   

date          2024-06-07                                                 \
           total_returns annualized_return market_value equity_invested   
Date                                                                      
2026-05-13        0.0579            0.0296   1,947.2200      1,872.2900   
2026-05-14        0.0693            0.0353   1,968.5601      1,872.2900   
2026-05-15        0.0533            0.0272   1,938.6401      1,872.2900   

date                                                   \
            benefit dividends_received dividend_yield   
Date                                                    
2026-05-13 108.3300            36.3000         0.0194   
2026-05-14 129.6701            36.3000         0.0194   
2026-05-15  99.7501            36.3000         0.0194   

date                                                                       \
           annualized_dividend_yield fractional_shares estimated_etf_fees   
Date                                                                        
2026-05-13                    0.0100            0.0000             0.0000   
2026-05-14                    0.0100            0.0000             0.0000   
2026-05-15                    0.0100            0.0000             0.0000   

date          2025-06-09                                                 \
           total_returns annualized_return market_value equity_invested   
Date                                                                      
2026-05-13       -0.0357           -0.0386   4,425.5001      4,585.3600   
2026-05-14       -0.0252           -0.0271   4,474.0002      4,585.3600   
2026-05-15       -0.0400           -0.0429   4,406.0001      4,585.3600   

date                                                    \
             benefit dividends_received dividend_yield   
Date                                                     
2026-05-13 -163.8428             0.0000         0.0000   
2026-05-14 -115.3864             0.0000         0.0000   
2026-05-15 -183.3253             0.0000         0.0000   

date                                                                       \
           annualized_dividend_yield fractional_shares estimated_etf_fees   
Date                                                                        
2026-05-13                    0.0000            0.0000             0.0000   
2026-05-14                    0.0000            0.0000             0.0000   
2026-05-15                    0.0000            0.0000             0.0000   

date          2025-10-31                                                 \
           total_returns annualized_return market_value equity_invested   
Date                                                                     

In [121]:
pd.concat(la.investments.loc[target].metrics['performance']).unstack(0).iloc[-1].unstack(0)

date,2024-02-05,2024-06-07,2025-06-09,2025-10-31,2026-05-12
total_returns,0.1830,0.0533,-0.0400,0.0455,-0.0010
annualized_return,0.0768,0.0272,-0.0429,0.0864,-0.1158
market_value,528.7200,"1,938.6401","4,406.0001","4,758.4801","5,992.1602"
equity_invested,502.1700,"1,872.2900","4,585.3600","4,547.3700","5,992.8200"
benefit,91.9100,99.7501,-183.3253,206.8275,-6.0528
dividends_received,19.5000,36.3000,0.0000,0.0000,0.0000
dividend_yield,0.0388,0.0194,0.0000,0.0000,0.0000
annualized_dividend_yield,0.0169,0.0100,0.0000,0.0000,0.0000
fractional_shares,47.7600,0.0000,0.0000,0.0000,0.0000
estimated_etf_fees,0.0000,0.0000,0.0000,0.0000,0.0000


In [122]:
aux = pd.concat(
    [pd.Series(inv.metrics['risk']) * 100 for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux

,AYV.PA,DCAM.PA,ESE.PA,AI.PA,PINR.PA,GLE.PA,TEP.PA,SU.PA
volatility,2.1718,0.9654,1.0347,1.4666,1.3171,2.5705,2.3210,2.0215
annualized_volatility,34.4765,15.3248,16.4248,23.2808,20.9082,40.8061,36.8449,32.0904
downside_deviation,1.5339,0.8186,0.8222,1.0366,1.0741,1.9138,1.8161,1.4405
max_drawdown,-59.1741,-15.4457,-33.6206,-39.3882,-41.6952,-87.6530,-87.0802,-61.7935
VaR,-3.2108,-1.2500,-1.6047,-2.2385,-1.8966,-3.7810,-3.3361,-3.1155
CVaR,-4.8529,-2.2705,-2.5305,-3.3245,-3.0971,-6.0379,-5.3881,-4.5620


In [123]:
aux = pd.concat(la.investments.loc[target].metrics['positions']).unstack(0).iloc[-1].unstack(0)
aux

date,2024-02-05,2024-06-07,2025-06-09,2025-10-31,2026-05-12
nominal_shares,3.0000,11.0000,25.0000,27.0000,34.0000
adjusted_shares,3.0000,11.0000,25.0000,27.0000,34.0000
long_value,166.5600,169.4364,182.5200,167.6000,175.4000
fees,0.9900,2.9000,4.1100,4.0700,5.3700
taxes,1.5000,5.5900,18.2500,18.1000,23.8500
subsidies,0.0000,0.0000,0.0000,0.0000,0.0000


In [124]:
perf = pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1, keys=[inv.name for inv in la.investments])
perf

AYV.PA    DCAM.PA     ESE.PA      AI.PA  \
date                      2023-10-31 2025-03-20 2024-12-09 2024-02-05   
total_returns                    NaN     0.2122     0.1197     0.1830   
annualized_return                NaN     0.1817     0.0823     0.0768   
market_value                     NaN 5,876.9999 4,852.1399   528.7200   
equity_invested                  NaN 4,844.0000 4,329.6350   502.1700   
benefit                          NaN 1,027.7106   518.1379    91.9100   
dividends_received               NaN     0.0000     0.0000    19.5000   
dividend_yield                   NaN     0.0000     0.0000     0.0388   
annualized_dividend_yield        NaN     0.0000     0.0000     0.0169   
fractional_shares                NaN     0.0000     0.0000    47.7600   
estimated_etf_fees            0.0000    12.3084     8.1304     0.0000   

                                                                       \
date                      2024-06-07 2025-06-09 2025-10-31 2026-05-12   
total_returns                 0.0533    -0.0400     0.0455    -0.0010   
annualized_return             0.0272    -0.0429     0.0864    -0.1158   
market_value              1,938.6401 4,406.0001 4,758.4801 5,992.1602   
equity_invested           1,872.2900 4,585.3600 4,547.3700 5,992.8200   
benefit                      99.7501  -183.3253   206.8275    -6.0528   
dividends_received           36.3000     0.0000     0.0000     0.0000   
dividend_yield                0.0194     0.0000     0.0000     0.0000   
annualized_dividend_yield     0.0100     0.0000     0.0000     0.0000   
fractional_shares             0.0000     0.0000     0.0000     0.0000   
estimated_etf_fees            0.0000     0.0000     0.0000     0.0000   

                             PINR.PA     GLE.PA     TEP.PA      SU.PA  
date                      2024-07-30 2023-10-13 2024-05-16 2025-06-09  
total_returns                -0.2548     2.1396    -0.3035     0.1861  
annualized_return            -0.1515     0.5561    -0.1657     0.2013  
market_value              1,452.4620 1,463.2200   627.1200 5,275.0000  
equity_invested           1,945.2840   483.3500 1,001.6400 4,514.0100  
benefit                    -495.7220 1,034.1700  -303.9700   840.2425  
dividends_received            0.0000    57.2000    72.4500    84.0000  
dividend_yield                0.0000     0.1183     0.0723     0.0186  
annualized_dividend_yield     0.0000     0.0442     0.0356     0.0200  
fractional_shares             0.0000     0.0000     0.0000     0.0000  
estimated_etf_fees           26.7444     0.0000     0.0000     0.0000

In [125]:
aux = la.investments.iloc[0]._inflation_rates
(aux/aux.iloc[0] * 100).plot()

c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [126]:
perf.T.groupby(level=0)['benefit'].sum()

AI.PA       209.1096
AYV.PA             0
DCAM.PA   1,027.7106
ESE.PA      518.1379
GLE.PA    1,034.1700
PINR.PA    -495.7220
SU.PA       840.2425
TEP.PA     -303.9700
Name: benefit, dtype: object

In [127]:
pd.concat([pd.concat(inv.metrics['positions']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1, keys=[inv.name for inv in la.investments])

AYV.PA    DCAM.PA     ESE.PA      AI.PA             \
date            2023-10-31 2025-03-20 2024-12-09 2024-02-05 2024-06-07   
nominal_shares         NaN 1,000.0000   150.0000     3.0000    11.0000   
adjusted_shares        NaN 1,000.0000   150.0000     3.0000    11.0000   
long_value             NaN     4.8440    28.8389   166.5600   169.4364   
fees                   NaN     0.0000     3.8000     0.9900     2.9000   
taxes                  NaN     0.0000     0.0000     1.5000     5.5900   
subsidies           0.0000     0.0000     0.0000     0.0000     0.0000   

                                                    PINR.PA     GLE.PA  \
date            2025-06-09 2025-10-31 2026-05-12 2024-07-30 2023-10-13   
nominal_shares     25.0000    27.0000    34.0000    66.0000    22.0000   
adjusted_shares    25.0000    27.0000    34.0000    66.0000    22.0000   
long_value        182.5200   167.6000   175.4000    29.4740    21.8600   
fees                4.1100     4.0700     5.3700     0.0000     0.9900   
taxes              18.2500    18.1000    23.8500     0.0000     1.4400   
subsidies           0.0000     0.0000     0.0000     0.0000     0.0000   

                    TEP.PA      SU.PA  
date            2024-05-16 2025-06-09  
nominal_shares      9.0000    20.0000  
adjusted_shares     9.0000    20.0000  
long_value        110.7500   224.6000  
fees                1.9000     4.0400  
taxes               2.9900    17.9700  
subsidies           0.0000     0.0000

In [128]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sort_index()

date
2023-10-13     483.3500
2023-10-31          NaN
2024-02-05     502.1700
2024-05-16   1,001.6400
2024-06-07   1,872.2900
2024-07-30   1,945.2840
2024-12-09   4,329.6350
2025-03-20   4,844.0000
2025-06-09   4,585.3600
2025-06-09   4,514.0100
2025-10-31   4,547.3700
2026-05-12   5,992.8200
Name: equity_invested, dtype: object

In [129]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sum()

34617.929

In [130]:
from utils.fees import fees
((pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sort_index()).apply(fees) + pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sort_index())

date
2023-10-13   1,035.1600
2023-10-31          NaN
2024-02-05      93.8100
2024-05-16    -301.0700
2024-06-07     102.6501
2024-07-30    -492.8220
2024-12-09     521.9379
2025-03-20   1,032.0702
2025-06-09    -179.1984
2025-06-09     844.3051
2025-10-31     210.9201
2026-05-12      -0.6592
dtype: object

In [131]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sort_index()

date
2023-10-13   1,034.1700
2023-10-31          NaN
2024-02-05      91.9100
2024-05-16    -303.9700
2024-06-07      99.7501
2024-07-30    -495.7220
2024-12-09     518.1379
2025-03-20   1,027.7106
2025-06-09    -183.3253
2025-06-09     840.2425
2025-10-31     206.8275
2026-05-12      -6.0528
Name: benefit, dtype: object

In [132]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sum()

2829.6786046548577

In [133]:
la.investments_closed_positions

,,value_adj_long,fees_adj_long,short_date,count_short,value_short,fees_short,fs_adj,tax_adj,abond_adj,indice
ticker,date,,,,,,,,,,
AYV.PA,2023-10-31,6.3300,0.9900,2025-02-06,78.0000,7.2100,1.9000,0.0000,1.4800,NaN,^FCHI


### Market value

In [134]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['market_value'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux.div(aux.sum(axis=1), axis=0)
aux

AYV.PA    DCAM.PA     ESE.PA      AI.PA                        \
date       2023-10-31 2025-03-20 2024-12-09 2024-02-05 2024-06-07 2025-06-09   
Date                                                                           
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-16        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-17        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-18        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-19        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2026-05-11        NaN 5,823.0000 4,778.4900   528.0000 1,936.0000 4,400.0000   
2026-05-12        NaN 5,787.0002 4,753.2300   527.4600 1,934.0201 4,395.5002   
2026-05-13        NaN 5,849.0000 4,802.1749   531.0600 1,947.2200 4,425.5001   
2026-05-14        NaN 5,926.0001 4,876.5902   536.8800 1,968.5601 4,474.0002   
2026-05-15        NaN 5,876.9999 4,852.1399   528.7200 1,938.6401 4,406.0001   

                                    PINR.PA     GLE.PA     TEP.PA      SU.PA  
date       2025-10-31 2026-05-12 2024-07-30 2023-10-13 2024-05-16 2025-06-09  
Date                                                                          
2023-10-13        NaN        NaN        NaN   479.7100        NaN        NaN  
2023-10-16        NaN        NaN        NaN   482.9000        NaN        NaN  
2023-10-17        NaN        NaN        NaN   479.4900        NaN        NaN  
2023-10-18        NaN        NaN        NaN   475.8600        NaN        NaN  
2023-10-19        NaN        NaN        NaN   470.6900        NaN        NaN  
...               ...        ...        ...        ...        ...        ...  
2026-05-11 4,752.0000        NaN 1,457.9400 1,526.5800   616.1400 5,492.9999  
2026-05-12 4,747.1402 5,977.8802 1,419.0000 1,469.1600   605.7000 5,305.9998  
2026-05-13 4,779.5401 6,018.6801 1,439.1960 1,463.4399   597.9600 5,367.9999  
2026-05-14 4,831.9202 6,084.6402 1,456.0261 1,484.1200   598.8600 5,419.0002  
2026-05-15 4,758.4801 5,992.1602 1,452.4620 1,463.2200   627.1200 5,275.0000  

[671 rows x 12 columns]

In [135]:
aux.sum(axis=1)

Date
2023-10-13      479.7100
2023-10-16      482.9000
2023-10-17      479.4900
2023-10-18      475.8600
2023-10-19      470.6900
                 ...    
2026-05-11   31,311.1498
2026-05-12   36,922.0907
2026-05-13   37,221.7712
2026-05-14   37,656.5972
2026-05-15   37,170.9423
Length: 671, dtype: object

### Total returns

In [136]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['total_returns'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux = aux * 100
aux

AYV.PA    DCAM.PA     ESE.PA      AI.PA                        \
date       2023-10-31 2025-03-20 2024-12-09 2024-02-05 2024-06-07 2025-06-09   
Date                                                                           
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-16        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-17        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-18        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-19        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2026-05-11        NaN    20.1024    10.2677    18.1592     5.1867    -4.1253   
2026-05-12        NaN    19.3599     9.6848    18.0517     5.0809    -4.2234   
2026-05-13        NaN    20.6386    10.8143    18.7685     5.7860    -3.5732   
2026-05-14        NaN    22.2268    12.5315    19.9275     6.9257    -2.5164   
2026-05-15        NaN    21.2162    11.9672    18.3026     5.3277    -3.9981   

                                    PINR.PA     GLE.PA     TEP.PA      SU.PA  
date       2025-10-31 2026-05-12 2024-07-30 2023-10-13 2024-05-16 2025-06-09  
Date                                                                          
2023-10-13        NaN        NaN        NaN    -0.9579        NaN        NaN  
2023-10-16        NaN        NaN        NaN    -0.2979        NaN        NaN  
2023-10-17        NaN        NaN        NaN    -1.0034        NaN        NaN  
2023-10-18        NaN        NaN        NaN    -1.7544        NaN        NaN  
2023-10-19        NaN        NaN        NaN    -2.8240        NaN        NaN  
...               ...        ...        ...        ...        ...        ...  
2026-05-11     4.4059        NaN   -25.2017   227.0673   -31.4434    23.4392  
2026-05-12     4.2991    -0.3391   -27.2034   215.1877   -32.4857    19.3002  
2026-05-13     5.0110     0.3411   -26.1652   214.0043   -33.2585    20.6725  
2026-05-14     6.1618     1.4408   -25.3001   218.2828   -33.1686    21.8013  
2026-05-15     4.5483    -0.1010   -25.4833   213.9588   -30.3472    18.6141  

[671 rows x 12 columns]

In [137]:
# Aplatir les colonnes multi-index
aux_flat = aux.copy()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]

# Maintenant tu peux plotter
aux_flat.plot()

c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



### Annualized returns

In [138]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['annualized_return'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux = aux * 100
aux

AYV.PA    DCAM.PA     ESE.PA      AI.PA                        \
date       2023-10-31 2025-03-20 2024-12-09 2024-02-05 2024-06-07 2025-06-09   
Date                                                                           
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-16        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-17        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-18        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-19        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2026-05-11        NaN    17.4030     7.1349     7.6576     2.6621    -4.4763   
2026-05-12        NaN    16.7237     6.7219     7.6047     2.6047    -4.5693   
2026-05-13        NaN    17.7696     7.4791     7.8829     2.9570    -3.8556   
2026-05-14        NaN    19.0704     8.6290     8.3355     3.5251    -2.7086   
2026-05-15        NaN    18.1667     8.2305     7.6767     2.7179    -4.2885   

                                    PINR.PA     GLE.PA     TEP.PA      SU.PA  
date       2025-10-31 2026-05-12 2024-07-30 2023-10-13 2024-05-16 2025-06-09  
Date                                                                          
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN  
2023-10-16        NaN        NaN        NaN   -30.4591        NaN        NaN  
2023-10-17        NaN        NaN        NaN   -60.1827        NaN        NaN  
2023-10-18        NaN        NaN        NaN   -72.5549        NaN        NaN  
2023-10-19        NaN        NaN        NaN   -82.5158        NaN        NaN  
...               ...        ...        ...        ...        ...        ...  
2026-05-11     8.5479        NaN   -15.0552    58.4006   -17.3196    25.7229  
2026-05-12     8.2919        NaN   -16.3172    56.0683   -17.9329    21.0782  
2026-05-13     9.6426   246.8973   -15.6277    55.7675   -18.3840    22.5146  
2026-05-14    11.8513 1,263.2690   -15.0540    56.5118   -18.3061    23.6757  
2026-05-15     8.6419   -11.5763   -15.1493    55.6128   -16.5728    20.1274  

[671 rows x 12 columns]

In [139]:
aux_flat = aux.copy()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]

for col in aux_flat.columns:

    mask = aux_flat[col].notna()

    if not mask.any():
        continue

    # position de la première valeur non-NaN
    start_pos = mask.values.argmax()

    # label d'index correspondant
    start_label = aux_flat.index[start_pos]

    # labels des 160 lignes suivantes
    end_pos = start_pos + 91
    end_pos = min(end_pos, len(aux_flat) - 1)
    end_label = aux_flat.index[end_pos]

    # mise à NaN avec loc
    aux_flat.loc[start_label:end_label, col] = np.nan

aux_flat.plot()

c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [140]:
la.market_data_manager._data['GLE.PA']

Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2000-01-03,51.5734,51.6851,49.2505,49.8089,16.9484,"1,078,214.0000",0.0000,0.0000
2000-01-04,49.5855,49.6749,48.7144,49.5855,16.8724,"2,529,529.0000",0.0000,0.0000
2000-01-05,48.6921,49.5855,48.7144,49.1388,16.7204,"1,233,422.0000",0.0000,0.0000
2000-01-06,48.8038,49.5855,48.5804,48.6921,16.5684,"1,221,423.0000",0.0000,0.0000
2000-01-07,48.4687,49.3622,48.4687,48.6921,16.5684,"866,415.0000",0.0000,0.0000
...,...,...,...,...,...,...,...,...
2026-05-11,69.1400,69.8300,68.9000,69.3900,69.3900,"956,469.0000",0.0000,0.0000
2026-05-12,67.9400,67.9600,66.1800,66.7800,66.7800,"1,912,563.0000",0.0000,0.0000
2026-05-13,67.3900,67.7200,65.7700,66.5200,66.5200,"1,382,576.0000",0.0000,0.0000


In [141]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['annualized_return'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux * 100

AYV.PA    DCAM.PA     ESE.PA      AI.PA                        \
date       2023-10-31 2025-03-20 2024-12-09 2024-02-05 2024-06-07 2025-06-09   
Date                                                                           
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-16        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-17        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-18        NaN        NaN        NaN        NaN        NaN        NaN   
2023-10-19        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2026-05-11        NaN    17.4030     7.1349     7.6576     2.6621    -4.4763   
2026-05-12        NaN    16.7237     6.7219     7.6047     2.6047    -4.5693   
2026-05-13        NaN    17.7696     7.4791     7.8829     2.9570    -3.8556   
2026-05-14        NaN    19.0704     8.6290     8.3355     3.5251    -2.7086   
2026-05-15        NaN    18.1667     8.2305     7.6767     2.7179    -4.2885   

                                    PINR.PA     GLE.PA     TEP.PA      SU.PA  
date       2025-10-31 2026-05-12 2024-07-30 2023-10-13 2024-05-16 2025-06-09  
Date                                                                          
2023-10-13        NaN        NaN        NaN        NaN        NaN        NaN  
2023-10-16        NaN        NaN        NaN   -30.4591        NaN        NaN  
2023-10-17        NaN        NaN        NaN   -60.1827        NaN        NaN  
2023-10-18        NaN        NaN        NaN   -72.5549        NaN        NaN  
2023-10-19        NaN        NaN        NaN   -82.5158        NaN        NaN  
...               ...        ...        ...        ...        ...        ...  
2026-05-11     8.5479        NaN   -15.0552    58.4006   -17.3196    25.7229  
2026-05-12     8.2919        NaN   -16.3172    56.0683   -17.9329    21.0782  
2026-05-13     9.6426   246.8973   -15.6277    55.7675   -18.3840    22.5146  
2026-05-14    11.8513 1,263.2690   -15.0540    56.5118   -18.3061    23.6757  
2026-05-15     8.6419   -11.5763   -15.1493    55.6128   -16.5728    20.1274  

[671 rows x 12 columns]

In [142]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['benefit'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
# Aplatir les colonnes multi-index
aux_flat = aux.copy().ffill()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]
aux_flat['Portfolio'] = aux_flat.sum(axis=1)
# Maintenant tu peux plotter
aux_flat.plot()

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_6380\305826210.py:7: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [143]:
aux_flat

,AYV.PA_2023-10-31 00:00:00,DCAM.PA_2025-03-20 00:00:00,ESE.PA_2024-12-09 00:00:00,AI.PA_2024-02-05 00:00:00,AI.PA_2024-06-07 00:00:00,AI.PA_2025-06-09 00:00:00,AI.PA_2025-10-31 00:00:00,AI.PA_2026-05-12 00:00:00,PINR.PA_2024-07-30 00:00:00,GLE.PA_2023-10-13 00:00:00,TEP.PA_2024-05-16 00:00:00,SU.PA_2025-06-09 00:00:00,Portfolio
Date,,,,,,,,,,,,,
2023-10-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.6300,NaN,NaN,-4.6300
2023-10-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.4400,NaN,NaN,-1.4400
2023-10-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.8500,NaN,NaN,-4.8500
2023-10-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-8.4800,NaN,NaN,-8.4800
2023-10-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-13.6500,NaN,NaN,-13.6500
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-11,101.3200,973.7593,444.5543,91.1900,97.1100,-189.1600,200.3532,NaN,-490.2440,"1,097.5300",-314.9500,"1,058.0462","3,069.5090"
2026-05-12,101.3200,937.7919,419.3171,90.6500,95.1301,-193.6598,195.4978,-20.3198,-529.1840,"1,040.1100",-325.3900,871.2144,"2,682.4776"
2026-05-13,101.3200,999.7359,468.2180,94.2500,108.3300,-163.8428,227.8685,20.4433,-508.9880,"1,034.3899",-333.1300,933.1587,"2,981.7536"


In [144]:
la.investments.iloc[2].metrics['risk_adjusted_performance']

{'sharpe_ratio': None,
 'sortino_ratio': None,
 'alpha': None,
 'beta': None,
 'information_ratio': None}

In [145]:
la.investments.iloc[2].transactions['annual_fee_rate'].values[0]

#.market_data['Close']

0.0013

In [146]:
i = 0
print(la.investments.iloc[i].name)
pd.concat(la.investments.iloc[i].metrics['performance']).unstack(0).xs('total_returns', axis=1, level=1).plot()

AYV.PA


c:\Users\Alexandre\AppData\Local\Programs\Python\Python39\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [147]:
la.investments.iloc[1].transactions

,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date,count_nm,count_adj,value_adj,tax_adj,abond_adj,fees_adj,fs_adj
date,,,,,,,,,,,,,,,,
2025-03-20,"1,000.0000",4.8440,4.3600,-4.3600,0.0020,0.0000,^GSPC,NaN,NaT,"1,000.0000","1,000.0000",4.8440,0.0000,NaN,0.0000,0.0000


In [148]:
la.investments.iloc[0].simulated_transactions

,value_adj,fees_adj,count_nm,count_adj,fs_adj,tax_adj,abond_adj,indice,empty_date
short_date,,,,,,,,,
2025-02-06,7.2100,0.9900,78.0000,78.0000,0.0000,1.4800,0,^FCHI,NaT


In [149]:
la.investments.iloc[0].simulated_positions_timeline

short_date 2025-02-06                                            
             count_nm count_adj value_adj fees_adj tax_adj fs_adj
Date                                                             
2025-02-06    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2025-02-07    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2025-02-10    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2025-02-11    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2025-02-12    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
...               ...       ...       ...      ...     ...    ...
2026-05-11    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2026-05-12    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2026-05-13    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2026-05-14    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000
2026-05-15    78.0000   78.0000    7.2100   0.9900  1.4800 0.0000

[329 rows x 6 columns]

In [150]:
la.investments.iloc[1].simulated_transactions

""


In [151]:
la.investments.iloc[1].simulated_positions_timeline

""


In [152]:
la.investments.iloc[1].transactions

,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date,count_nm,count_adj,value_adj,tax_adj,abond_adj,fees_adj,fs_adj
date,,,,,,,,,,,,,,,,
2025-03-20,"1,000.0000",4.8440,4.3600,-4.3600,0.0020,0.0000,^GSPC,NaN,NaT,"1,000.0000","1,000.0000",4.8440,0.0000,NaN,0.0000,0.0000


In [153]:
df = la.investments.iloc[0].transactions
df

,count,value,broker_fees,other_fees,annual_fee_rate,tax,indice,abondement,empty_date,count_nm,count_adj,value_adj,tax_adj,abond_adj,fees_adj,fs_adj
date,,,,,,,,,,,,,,,,
2023-10-31,78.0000,6.3300,0.9900,0.0000,NaN,1.4800,^FCHI,NaN,2025-02-06,0.0000,0.0000,6.3300,0.0000,NaN,0.0000,0.0000
2025-02-06,-78.0000,7.2100,1.9000,0.0000,NaN,0.0000,^FCHI,NaN,NaT,-78.0000,0.0000,7.2100,0.0000,NaN,1.9000,0.0000


In [154]:
hjn

NameError: name 'hjn' is not defined

In [ ]:
df_port = la.investments.iloc[0].positions_timeline
idx = pd.IndexSlice  # Pour manipuler MultiIndex proprement
df_port.loc[:, idx[:, 'count_nm']].droplevel(1, axis=1)

In [ ]:
def fees(x: float) -> float:
    """
    Returns the transaction fee charged by Bourse Direct for a PEA active_group_posount.

    The fee depends on the transaction amount based on predefined brackets:
    - Up to €500: €0.99
    - €501 to €1000: €1.90
    - €1001 to €2000: €2.90
    - €2001 to €4400: €3.80
    - Above €4400: 0.09% of the amount

    Args:
        x (float): The amount of the transaction (buy or sell)

    Returns:
        float: The corresponding fee in euros

    Examples:
        >>> fees(500)
        0.99
        >>> fees(1000)
        1.9
        >>> fees(2500)
        3.8
        >>> fees(6000)
        5.4
    """
    if x <= 500:
        return 0.99
    elif x <= 1000:
        return 1.9
    elif x <= 2000:
        return 2.9
    elif x <= 4400:
        return 3.80
    else:
        return x * 0.0009

df_port = la.investments.iloc[0].positions_timeline


In [ ]:
def compute_performance_metrics(
    self, 
    fees_func=None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Compute historical performance metrics for this listed investment.

    This function calculates key time-series metrics per asset, including:
        - Nominal and adjusted share counts
        - Fractional shares
        - Market values and long values
        - Fees and taxes
        - Dividends
        - Equity invested
        - Asset-level returns and contribution to portfolio performance

    Parameters
    ----------
    fees_func : callable, optional
        Function to compute fees on market value. Signature: `fees_func(value: float) -> float`.
        If None, fees are assumed to be included in `fees_adj`.

    Returns
    -------
    tuple[pd.DataFrame, ...]
        df_nominal : Nominal share quantities ('count_nm')
        df_adjusted : Adjusted share quantities ('count_adj')
        df_fractional : Fractional shares ('fs_adj')
        df_long_value : Transaction-level value ('value_adj')
        df_fees : Fees paid ('fees_adj')
        df_taxes : Taxes paid ('tax_adj')
        df_market_value : Market value of positions
        df_equity : Equity invested (including fees/taxes)
        df_dividends : Cumulative dividends
        df_contribution : Asset-level performance contribution
        df_returns : Returns per asset (benefit / equity)
    """
    idx = pd.IndexSlice

    # Extract columns from investment timeline
    df_nominal = self.positions_timeline.loc[:, idx[:, "count_nm"]].droplevel(1, axis=1)
    df_adjusted = self.positions_timeline.loc[:, idx[:, "count_adj"]].droplevel(1, axis=1)
    df_fractional = self.positions_timeline.loc[:, idx[:, "fs_adj"]].droplevel(1, axis=1)
    df_long_value = self.positions_timeline.loc[:, idx[:, "value_adj"]].droplevel(1, axis=1)
    df_fees = self.positions_timeline.loc[:, idx[:, "fees_adj"]].droplevel(1, axis=1)
    df_taxes = self.positions_timeline.loc[:, idx[:, "tax_adj"]].droplevel(1, axis=1)

    # Market prices and dividends
    df_prices = self.stock_data["Close"]
    df_divs = self.stock_data["Dividends"].fillna(0.0)

    # Compute market value and equity invested
    df_market_value = df_adjusted.mul(df_prices, axis=0)
    df_equity = df_long_value * df_adjusted + df_fees + df_taxes

    # Compute cumulative dividends
    df_cumulative_divs = (df_nominal * df_divs).cumsum()

    # Compute total asset benefit
    if fees_func is None:
        fees_func = lambda x: 0.0  # assume fees are already in df_fees

    df_benefit = (
        df_market_value
        + df_cumulative_divs
        + df_fractional
        - df_market_value.applymap(fees_func)
        - df_equity
    )

    # Asset-level returns
    df_returns = df_benefit.div(df_equity.replace(0, np.nan))

    return (
        df_nominal,
        df_adjusted,
        df_fractional,
        df_long_value,
        df_fees,
        df_taxes,
        df_market_value,
        df_equity,
        df_cumulative_divs,
        df_benefit,
        df_returns,
    )


In [ ]:
import numpy as np

def compute_performance_metrics(
    self, 
    fees_func=None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Compute historical performance metrics for this listed investment.

    This function calculates key time-series metrics per asset, including:
        - Nominal and adjusted share counts
        - Fractional shares
        - Market values and long values
        - Fees and taxes
        - Dividends
        - Equity invested
        - Asset-level returns and contribution to portfolio performance

    Parameters
    ----------
    fees_func : callable, optional
        Function to compute fees on market value. Signature: `fees_func(value: float) -> float`.
        If None, fees are assumed to be included in `fees_adj`.

    Returns
    -------
    tuple[pd.DataFrame, ...]
        df_nominal : Nominal share quantities ('count_nm')
        df_adjusted : Adjusted share quantities ('count_adj')
        df_fractional : Fractional shares ('fs_adj')
        df_long_value : Transaction-level value ('value_adj')
        df_fees : Fees paid ('fees_adj')
        df_taxes : Taxes paid ('tax_adj')
        df_market_value : Market value of positions
        df_equity : Equity invested (including fees/taxes)
        df_dividends : Cumulative dividends
        df_contribution : Asset-level performance contribution
        df_returns : Returns per asset (benefit / equity)
    """
    idx = pd.IndexSlice

    # Extract columns from investment timeline
    df_nominal = self.positions_timeline.loc[:, idx[:, "count_nm"]].droplevel(1, axis=1)
    df_adjusted = self.positions_timeline.loc[:, idx[:, "count_adj"]].droplevel(1, axis=1)
    df_fractional = self.positions_timeline.loc[:, idx[:, "fs_adj"]].droplevel(1, axis=1)
    df_long_value = self.positions_timeline.loc[:, idx[:, "value_adj"]].droplevel(1, axis=1)
    df_fees = self.positions_timeline.loc[:, idx[:, "fees_adj"]].droplevel(1, axis=1)
    df_taxes = self.positions_timeline.loc[:, idx[:, "tax_adj"]].droplevel(1, axis=1)

    # Market prices and dividends
    df_prices = self.stock_data["Close"]
    df_divs = self.stock_data["Dividends"].fillna(0.0)

    # Compute market value and equity invested
    df_market_value = df_adjusted.mul(df_prices, axis=0)
    df_equity = df_long_value * df_adjusted + df_fees + df_taxes

    # Compute cumulative dividends
    df_cumulative_divs = (df_nominal * df_divs).cumsum()

    # Compute total asset benefit
    if fees_func is None:
        fees_func = lambda x: 0.0  # assume fees are already in df_fees

    df_benefit = (
        df_market_value
        + df_cumulative_divs
        + df_fractional
        - df_market_value.applymap(fees_func)
        - df_equity
    )

    # Asset-level returns
    df_returns = df_benefit.div(df_equity.replace(0, np.nan))

    return (
        df_nominal,
        df_adjusted,
        df_fractional,
        df_long_value,
        df_fees,
        df_taxes,
        df_market_value,
        df_equity,
        df_cumulative_divs,
        df_benefit,
        df_returns,
    )


In [ ]:
"""
Compute historical portfolio and per-asset performance metrics, including returns, weights, dividends, and valuations.

Args:
    df_port (pd.DataFrame): Vectorized data of positions and metrics.

Returns:
    tuple of pd.DataFrames and pd.Series:
        - df_cn: Nominal share quantities.
        - df_c: Adjusted share quantities.
        - df_fs: Fractional shares.
        - df_lv: Long value of positions.
        - df_f: Fees.
        - df_t: Taxes.
        - df_p: Prices.
        - df_d: Dividends.
        - df_v: Market value.
        - df_w: Weights in portfolio.
        - df_e: Equity invested.
        - df_r: Returns per position.
        - df_benef: Value contribution per asset.
        - df_wr: Weighted returns.
        - portfolio_cum_return: Total cumulative return of portfolio.
"""

idx = pd.IndexSlice  # Pour manipuler MultiIndex proprement
df_cn = df_port.loc[:, idx[:, 'count_nm']].droplevel(1, axis=1)
df_c = df_port.loc[:, idx[:, 'count_adj']].droplevel(1, axis=1)
df_fs = df_port.loc[:, idx[:, 'fs_adj']].droplevel(1, axis=1)
df_lv = df_port.loc[:, idx[:, 'value_adj']].droplevel(1, axis=1) # Pour long value
df_f = df_port.loc[:, idx[:, 'fees_adj']].droplevel(1, axis=1)
df_t = df_port.loc[:, idx[:, 'tax_adj']].droplevel(1, axis=1)

df_p = la.investments.iloc[0].stock_data['Close']
df_d = la.investments.iloc[0].stock_data['Dividends']

df_v = df_cn.mul(df_p, axis=0)
df_e = df_lv*df_c + df_f + df_t
df_div = df_cn.mul(df_d, axis=0).cumsum()
df_benef = df_v + df_div + df_fs - df_v.apply(lambda stock: stock.apply(lambda val: fees(val))) - df_e
df_r = df_benef / df_e

df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)

df_r

In [ ]:
df_cn.mul(df_d, axis=0).cumsum()

### Niveau portefeuille

In [ ]:
df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)
portfolio_cum_return

In [ ]:
df_v = df_p * df_cn
df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_e = df_lv*df_c + df_f + df_t
df_r = (df_v + (df_d*df_cn).cumsum() + df_fs - (df_v).apply(lambda stock: stock.apply(lambda val: fees(val))) - df_e) / df_e
df_benef = df_r.mul(df_e, axis=1)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)
df_benef

### Test

In [ ]:
from core.market_data.manager import MarketDataManager


In [ ]:
md = MarketDataManager(['AI.PA', 'AAPL', 'GLE.PA', 'AYV.PA', 'SU.PA', 'TEP.PA', 'DCAM.PA'])

In [ ]:
md.get_market_data('ESE.PA')

In [ ]:
df_data